query 和 key 经过注意力评分函数是输出结果经过 softmax 运算转化为 value 的概率分布

注意力汇聚的输出就是 value 基于这些注意力权重的加权和

**掩码 Softmax 操作**

普通softmax会对全部位置计算概率
但很多场景存在无效填充位置（无意义的特殊 token，未来不可见 token 等）

1. 把超出有效长度的无效位置打分替换成极小负数
2. 再做 softmax，极小负数经过指数后几乎等于 0

In [9]:
import matplotlib.pyplot as plt

def show_heatmaps(matrices, xlabel='Query', ylabel='Key', titles=None, figsize=(2.5, 2.5), cmap='Reds'):
    """
    可视化注意力权重热力图，直观体现：每一个查询，分别给每一个键分配了多大的关注度
    Args:
        matrices: 4维张量 [画布行数, 画布列数, num_queries, num_keys]
        titles: 子图标题列表
        figsize: 单张子图尺寸
        cmap: 热力配色
    """
    num_rows, num_cols = matrices.shape[0], matrices.shape[1]
    fig, axes = plt.subplots(
        num_rows, num_cols,
        figsize=(figsize[0]*num_cols, figsize[1]*num_rows),
        sharex=True, sharey=True,
        squeeze=False
    )

    pcm = None
    for i, (row_axes, row_mats) in enumerate(zip(axes, matrices)):
        for j, (ax, mat) in enumerate(zip(row_axes, row_mats)):
            data = mat.detach().cpu().numpy()
            pcm = ax.imshow(data, cmap=cmap)
            # 仅底行显示x标签
            if i == num_rows - 1:
                ax.set_xlabel(xlabel)
            # 仅首列显示y标签
            if j == 0:
                ax.set_ylabel(ylabel)
            # 设置子图标题
            if titles is not None:
                ax.set_title(titles[j])
    # 全局色条
    fig.colorbar(pcm, shrink=0.6)
    plt.tight_layout()
    plt.show()

In [7]:
import math
import torch
import torch.nn as nn


def sequence_mask(x: torch.Tensor, valid_lens: torch.Tensor, mask_value: float = -1e6) -> torch.Tensor:
    """
    对二维张量每行做尾部掩码：索引 >= valid_len 全部填充 mask_value
    Args:
        x: shape [N, L]，N行序列，每行长度L
        valid_lens: shape [N]，每行对应的有效长度
        mask_value: 掩码填充极小值
    Returns:
        掩码后的同shape张量
    """
    n_seq, seq_len = x.shape
    # 生成位置索引 0,1,...,seq_len-1
    pos = torch.arange(seq_len, device=x.device, dtype=torch.long)
    # 广播构造掩码：[N, L]，True代表需要掩码（超出有效长度）
    mask = pos.unsqueeze(0) >= valid_lens.unsqueeze(1)
    # 原地替换掩码区域
    x = x.masked_fill(mask, mask_value)
    return x


def masked_softmax(X: torch.Tensor, valid_lens: torch.Tensor = None) -> torch.Tensor:
    """
    带有效长度掩码的Softmax，屏蔽padding无效位置
    Args:
        X: 打分张量 shape [batch, n_query, n_key]
        valid_lens:
            - None: 不做掩码，标准softmax
            - 1D tensor [batch]: 同batch内所有query共享同一个有效长度
            - 2D tensor [batch, n_query]: 每个query独立有效长度
    Returns:
        attention_weights: shape 和输入X一致，每行softmax归一化，掩码位置权重≈0
    """
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)

    batch_size, n_q, n_k = X.shape
    # 统一 valid_lens 到一维 [batch * n_q]，每条query对应一个长度
    if valid_lens.dim() == 1:
        # [B] -> [B * n_q]，每个batch的长度复制n_q份
        valid_lens = torch.repeat_interleave(valid_lens, repeats=n_q)
    elif valid_lens.dim() == 2:
        # [B, n_q] -> [B * n_q]
        valid_lens = valid_lens.reshape(-1)
    else:
        raise ValueError(f"valid_lens仅支持1/2维，当前维度:{valid_lens.dim()}")

    # 保证长度张量和X同设备
    valid_lens = valid_lens.to(X.device)

    # 3D -> 2D [B*n_q, n_k] 批量掩码
    x_2d = X.reshape(-1, n_k)
    x_masked = sequence_mask(x_2d, valid_lens, mask_value=-1e6)

    # 恢复原始三维并softmax
    x_3d = x_masked.reshape(batch_size, n_q, n_k)
    return nn.functional.softmax(x_3d, dim=-1)

Bahdanau 注意力的评分函数：
$$
a(q,k) = w_v^\top \tanh\big(W_q q + W_k k\big)
$$

- $W_q$: [h,q]
- $W_k$: [h,k]
- $w_v$: h

In [4]:
class AdditiveAttention(nn.Module):
    """加性注意力"""

    def __init__(self, key_size, query_size, num_hiddens, dropout, **kwargs):
        super(AdditiveAttention, self).__init__(**kwargs)
        # 键投影层 k_dim → h_dim，无偏置
        self.W_k = nn.Linear(key_size, num_hiddens, bias=False)
        # 查询投影层 q_dim → h_dim，无偏置
        self.W_q = nn.Linear(query_size, num_hiddens, bias=False)
        # 打分输出层 h_dim → 1，输出单个相似度分数
        self.w_v = nn.Linear(num_hiddens, 1, bias=False)
        # dropout，作用在注意力权重上，正则
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens):
        # 1. 分别线性投影 Query、Key 到隐藏维度 num_hiddens
        queries, keys = self.W_q(queries), self.W_k(keys)
        # queries: (B, n_q, h)
        # keys:     (B, n_k, h)

        # 2. 广播相加，实现每个Query和全部Key融合
        # queries.unsqueeze(2) → (B, n_q, 1, h)
        # keys.unsqueeze(1)   → (B, 1, n_k, h)
        # 相加广播后：(B, n_q, n_k, h)
        features = queries.unsqueeze(2) + keys.unsqueeze(1)
        features = torch.tanh(features)  # 激活

        # 3. 单层线性层压缩为相似度分数
        # w_v: h → 1
        scores = self.w_v(features).squeeze(-1)
        # squeeze去掉最后一维，scores形状：(B, n_q, n_k)
        # 含义：第b批，第q个查询，对第k个键的原始打分

        # 4. 带掩码softmax，得到归一化注意力权重
        self.attention_weights = masked_softmax(scores, valid_lens)
        # attention_weights: (B, n_q, n_k) 每行和为1

        # 5. dropout + 批量矩阵乘法 bmm 注意力汇聚
        # dropout随机屏蔽部分权重，正则化
        # bmm( (B,n_q,n_k), (B,n_k,v_dim) ) → (B,n_q,v_dim)
        return torch.bmm(self.dropout(self.attention_weights), values)

In [10]:
class DotProductAttention(nn.Module):
    """缩放点积注意力"""

    def __init__(self, dropout, **kwargs):
        super().__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    # queries: (batch_size, 查询数量 n_q, 特征维度 d)
    # keys:    (batch_size, 键值对数 n_k, 特征维度 d)
    # values:  (batch_size, 键值对数 n_k, value_dim)
    # valid_lens: 一维[batch] / 二维[batch, n_q]，传入masked_softmax屏蔽无效key
    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]  # 获取 Q/K 公共维度 d

        # 1. 批量点积打分 + 缩放
        # keys.transpose(1,2): [B, n_k, d] → [B, d, n_k]
        # bmm(Q, K.T) = [B, n_q, d] @ [B, d, n_k] = [B, n_q, n_k]
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
    
        # 2. 掩码softmax得到归一化注意力权重
        self.attention_weights = masked_softmax(scores, valid_lens)
        
        # 3. dropout + 加权汇聚Value
        return torch.bmm(self.dropout(self.attention_weights), values)